In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from snsphd.viz import phd_style

from util.pcr_loader import PcrLoader

colors, swatches = phd_style(jupyterStyle=True)

SHOW_DARK = True
X_LIMITS = None
Y_LIMITS = None

DATASETS = {
    "35 um": {
        "path": "../data/DC_Pa250528a_60nm_hero/35um/6_1snap_w0.06_l50_35micron_1.9thermal_start_265_end_XXX.csv",
        "threshold": 0.010,
    },
    "42 um": {
        "path": "../data/DC_Pa250528a_60nm_hero/42um/6_1snap_w0.06_l50_42um_2.2thermal_start_261_end_XXX.csv",
        "threshold": 0.010,
    },
    "46 um (QCL)": {
        # "path": "../data/DC_Pa250528a_60nm_hero/46um_QCL/2.11.2026/qcl_30Hz_280mA_30.0V__0.002_time_10s_x6cycles_start_258mK.csv",
        "path": "../data/DC_Pa250528a_60nm_hero/46um_QCL/3.25.2026/46um_40Hz_30.7V_262mA_267mK_positivepulse_start_100us_end_300us_total_QCL_350us_3.0sx10cycle.csv",
        "threshold": 0.010,
    },
    "63 um (QCL)": {
        "path": "../data/DC_Pa250528a_60nm_hero/63um_QCL/3.26.2026/63um_160Hz_6.34V_27.2mA_252mK_positivepulse_start_100us_end_300us_total_QCL_350us_3.0sx4cycle.csv",
        "threshold": 0.010,
    },
}

PLOT_COLORS = {
    "35 um": plt.cm.plasma(0.15),
    "42 um": plt.cm.plasma(0.40),
    "46 um (QCL)": plt.cm.plasma(0.65),
    "63 um (QCL)": plt.cm.plasma(0.90),
}
MARKERS = {
    "35 um": "o",
    "42 um": "^",
    "46 um (QCL)": "D",
    "63 um (QCL)": "s",
}

In [ ]:
# Load each CSV once so threshold maps are easy to inspect.
datasets = {label: PcrLoader.from_threshold_csv(spec["path"]) for label, spec in DATASETS.items()}

for label, dataset in datasets.items():
    print(label, dataset.threshold_map())


In [ ]:
AUTO_SCALE = True  # True = normalize all curves to 1.0 at NORMALIZE_BIAS; False = use manual SCALE_FACTOR
NORMALIZE_BIAS = 0.09  # uA – only used when AUTO_SCALE is True

MANUAL_SCALE_FACTOR = {
    "35 um": 1.0,
    "42 um": 0.85,
    "46 um (QCL)": 0.59,
    "63 um (QCL)": 0.71,
}

OFFSET_X = {
    "35 um": 0,
    "42 um": 0,
    "46 um (QCL)": 0,
    "63 um (QCL)": 0,
}

OFFSET_Y = {
    "35 um": 0,
    "42 um": 0,
    "46 um (QCL)": 0,
    "63 um (QCL)": 0,
}

CROP_END = {
    "35 um": -5,
    "42 um": -4,
    "46 um (QCL)": -1,
    "63 um (QCL)": -1,
}

# These manual tweaks align the four device curves on one figure.
curves = {
    label: datasets[label].get_curve(spec["threshold"], label=label)
    for label, spec in DATASETS.items()
}

# Compute scale factors
if AUTO_SCALE:
    SCALE_FACTOR = {}
    for label, curve in curves.items():
        val_at_norm = np.interp(NORMALIZE_BIAS, curve.bias_uA, curve.counts)
        SCALE_FACTOR[label] = 1.0 / val_at_norm if val_at_norm != 0 else 1.0
        print(f"{label}: counts@{NORMALIZE_BIAS} uA = {val_at_norm:.1f}, scale = {SCALE_FACTOR[label]:.6f}")
else:
    SCALE_FACTOR = MANUAL_SCALE_FACTOR


fig, ax = plt.subplots(figsize=(7.5, 4.5))

for label, curve in curves.items():
    ax.plot(
        curve.bias_uA[:CROP_END[label]] + OFFSET_X[label],
        (curve.counts[:CROP_END[label]] + OFFSET_Y[label]) * SCALE_FACTOR[label],
        marker=MARKERS[label],
        color=PLOT_COLORS[label],
        linewidth=2.0,
        markersize=3,
        alpha=0.9,
        label=f"{label} | {curve.threshold.label}",
    )
    if SHOW_DARK and curve.dark_counts is not None:
        ax.plot(
            curve.bias_uA[: CROP_END[label]] + OFFSET_X[label],
            (curve.dark_counts[: CROP_END[label]] + OFFSET_Y[label]) * SCALE_FACTOR[label],
            marker="x",
            color=PLOT_COLORS[label],
            linewidth=1.8,
            markersize=3,
            alpha=0.6,
            linestyle="--",
            label=f"DCR {label}",
        )

# if AUTO_SCALE:
#     ax.axvline(NORMALIZE_BIAS, color="gray", linestyle=":", linewidth=1, alpha=0.5)
ax.set_xlabel(r"Bias Current ($\mu$A)")
ax.set_ylabel("Normalized Count Rate" if AUTO_SCALE else "Photon Count Rate (cps)")
if X_LIMITS is not None:
    ax.set_xlim(*X_LIMITS)
if Y_LIMITS is not None:
    ax.set_ylim(*Y_LIMITS)
ax.legend(frameon=False, loc="upper left", fontsize=10)
ax.grid(alpha=0.2)

if AUTO_SCALE:
    ax.set_ylim(0, 1.2)
else:

    ax.set_ylim(0, 40000)

plt.tight_layout()

out_dir = Path("../out/60nm_hero")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "hero_35_42_46_63um_counts.png", dpi=300)
fig.savefig(out_dir / "hero_35_42_46_63um_counts.pdf", dpi=300)
plt.show()

In [ ]:
import json
import re

DOWNSAMPLE = 4  # average every N points
X_OFFSET = 1000  # us — this value maps to 0

hist_dir = Path("../data/DC_Pa250528a_60nm_hero/46um_QCL/3.192026")
hist_files = sorted(hist_dir.glob("*.json"))[::2]

fig, ax = plt.subplots(figsize=(8, 4.8))

for i, path in enumerate(hist_files):
    color = plt.cm.viridis(i / max(len(hist_files) - 1, 1))

    m = re.search(r"_on_(\d+mA_[\d.]+V)_", path.name)
    label = m.group(1).replace("_", ", ") if m else path.stem

    with path.open() as fh:
        payload = json.load(fh)

    x_us = np.asarray(payload["x_axis_ps"], dtype=float) / 1e6
    y_counts = np.asarray(payload["histogram_counts"], dtype=float)

    n = (len(x_us) // DOWNSAMPLE) * DOWNSAMPLE
    x_us = x_us[:n].reshape(-1, DOWNSAMPLE).mean(axis=1) - X_OFFSET
    y_counts = y_counts[:n].reshape(-1, DOWNSAMPLE).mean(axis=1)

    ax.plot(x_us, y_counts, linewidth=1.5, alpha=0.9, label=label, color=color)

ax.set_xlim(-300, 800)
ax.set_xlabel("Time (µs)")
ax.set_ylabel("Histogram Counts")
ax.set_title("46 µm QCL Histogram Overlay")
ax.grid(alpha=0.2)
ax.legend(frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

csv_path = "../data/DC_Pa250528a_60nm_hero/46um_QCL/3.27.2026/testing_positive.csv"
df = pd.read_csv(csv_path, skiprows=13)

# Pull the three TL1 columns (column names include the threshold value in parens)
bias = df["Bias_Current"]
counts_tl1_col = [c for c in df.columns if c.startswith("Counts_TL1")][0]
counts_on_dcr_col = [c for c in df.columns if c.startswith("CountsOnDcr_TL1")][0]
dcounts_col = [c for c in df.columns if c.startswith("DCounts_TL1")][0]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    bias * 1e3,
    df[counts_tl1_col] - df[counts_on_dcr_col],
    label="CR - DCR_on",
    marker="o",
    markersize=3,
    linewidth=1.5,
)
ax.plot(
    bias * 1e3,
    df[counts_tl1_col] - df[dcounts_col],
    label="CR - DCR_off",
    marker="s",
    markersize=3,
    linewidth=1.5,
    linestyle="--",
)
# ax.plot(
#     bias * 1e3,
#     df[dcounts_col],
#     label="DCounts_TL1",
#     marker="^",
#     markersize=3,
#     linewidth=1.5,
#     linestyle=":",
# )


ax.set_xlabel("Bias Current (mA)")
ax.set_ylabel("Count Rate (cps)")
ax.set_title("46 µm QCL — testing_positive (3.27.2026)")
ax.legend(frameon=False)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
def load_power_ramp(path):
    """Load a power ramp CSV, auto-detecting the number of header rows."""
    with open(path) as fh:
        lines = fh.readlines()
    skip = next(i for i, line in enumerate(lines) if line.startswith("QCL_Current_mA"))
    return pd.read_csv(path, skiprows=skip)


RAMPS = {
    "3.19 run": "../data/DC_Pa250528a_60nm_hero/46um_QCL/3.192026/power_ramp_start_29.7V_0.1Vincrements_262mK_46um_QCL.csv",
    "2.10 run": "../data/DC_Pa250528a_60nm_hero/46um_QCL/2.10.2026/power_ramp_250mA_start_5s_2.csv",
}
RAMP_COLORS = {"3.19 run": colors["blue"], "2.10 run": colors["red"]}

ramp_data = {label: load_power_ramp(path) for label, path in RAMPS.items()}

fig, ax = plt.subplots(figsize=(7, 5))

for label, rdf in ramp_data.items():
    color = RAMP_COLORS[label]
    ax.plot(
        rdf["QCL_Current_mA"],
        rdf["Signal_Hz"],
        label=label,
        marker="o",
        markersize=3,
        linewidth=1.5,
        color=color,
    )
    ax.errorbar(
        rdf["QCL_Current_mA"],
        y=rdf["Signal_Hz"],
        yerr=10 * np.sqrt(np.abs(rdf["Signal_Hz"])),
        xerr=1.5,
        capsize=0,
        alpha=0.4,
        color=color,
        elinewidth=4,
    )

ax.legend(frameon=False)
ax.grid()
ax.set_xlabel("QCL Current (mA)")
ax.set_ylabel("SNSPD Count Rate (Hz)")

fig.savefig("../out/60nm_hero/qcl_power_ramp_signal.png", dpi=300)

In [ ]:
# small change to check how other collaboration computer responds with nbstripout and git diff.